In [ ]:
import re
import pandas as pd
from PyPDF2 import PdfReader

<strong>Ekstraksi PDF UNAIR IS</strong>

In [ ]:
# 1. Baca semua teks dari PDF
def read_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"
    return full_text

# 2. Ekstrak seluruh blok matkul
def extract_all_courses(text):
    blocks = re.split(r"Module designation\s+", text, flags=re.IGNORECASE)
    data = []

    for block in blocks[1:]:  # blok pertama sebelum matkul pertama di-skip
        lines = block.strip().splitlines()
        nama_matkul = lines[0].strip()

        # Capaian Pembelajaran
        capaian_match = re.search(
            r"(?:objectives/intended\s+learning outcomes|Learning Outcomes)\s+(.*?)\n(?:Content|This course discusses)",
            block,
            re.IGNORECASE | re.DOTALL
        )
        capaian = capaian_match.group(1).strip().replace('\n', ' ') if capaian_match else ""

        # Konten Mata Kuliah
        konten_match = re.search(
            r"(?:Content|This course discusses)\s*[:\-]?\s*(.*?)(?:\nExamination forms|\nFinal score|\nStudy and examination)",
            block,
            re.IGNORECASE | re.DOTALL
        )
        konten = konten_match.group(1).strip().replace('\n', ' ') if konten_match else ""

        data.append({
            "Course Title": nama_matkul,
            "Learning Outcomes": capaian,
            "Course Description": "",  # dikosongkan
            "Course Content": konten
        })
    
    return data

# 3. Eksekusi
pdf_path = r"C:\Users\asus\Downloads\wlewle\Capaian Pembelajaran\UNAIR_IS.pdf"  # pastikan path benar
text = read_pdf_text(pdf_path)
data = extract_all_courses(text)

# 4. Simpan ke Excel
df = pd.DataFrame(data)
df.to_excel(r"C:\Users\asus\Downloads\wlewle\Data\UNAIR_IS.xlsx", index=False)

print(f"✅ Berhasil menyimpan ke 'UNAIR_IS.xlsx'")

✅ Berhasil menyimpan ke 'UNAIR_IS.xlsx'


<strong>Ekstraksi PDF ITS IS</strong>

In [ ]:
import pdfplumber
import re
import pandas as pd

# Baca semua teks menjadi satu string panjang
pdf_path = r"C:\Users\asus\Downloads\wlewle\Capaian Pembelajaran\ITS_IS.pdf" 
with pdfplumber.open(pdf_path) as pdf:
    full_text = "\n".join([p.extract_text() for p in pdf.pages if p.extract_text()])

# Bersihkan spasi tab dll
full_text = re.sub(r'[ \t]+', ' ', full_text)

# Pola regex untuk mengambil semua bagian dari satu matkul
pattern = re.compile(
    r'Matakuliah\s*\n\s*(.*?)\s*'                                # Module Name
    r'Kode\s*:\s*\w+.*?'                                         # Skip kode/SKS
    r'Deskripsi Matakuliah\s*(.*?)\s*'                           # Description
    r'Capaian Pembelajaran Matakuliah\s*(.*?)\s*'                # Learning Outcomes
    r'Tujuan Pembelajaran yang Spesifik\s*.*?'                   # Skip tujuan
    r'Pokok Bahasan\s*(.*?)\s*Pustaka Utama',                    # Content (BETWEEN Tujuan & Pustaka)
    re.DOTALL
)

matches = pattern.findall(full_text)

# Simpan hasilnya
data = []
for m in matches:
    data.append({
        "Module Name": m[0].strip(),
        "Description": m[1].strip(),
        "Learning Outcomes": m[2].strip(),
        "Content": m[3].strip()
    })

# Export ke Excel
df = pd.DataFrame(data)
df = df.drop_duplicates(subset="Module Name")
df.to_excel("ITS_IS_Indo.xlsx", index=False)


Cannot set gray non-stroke color because /'P21' is an invalid float value


In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
import time

# Baca file Excel
file_path = r"C:\Users\asus\Downloads\wlewle\ninuninu\ITS_IS_Indo.xlsx"  # Ubah jika file di path berbeda
df = pd.read_excel(file_path)

# Kolom-kolom yang akan diterjemahkan
kolom_terjemahkan = ["Module Name", "Description", "Learning Outcomes", "Content"]

# Fungsi untuk menerjemahkan teks
def translate_text(text):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return text
    try:
        translated = GoogleTranslator(source='id', target='en').translate(text)
        time.sleep(1)  # jeda untuk menghindari batas kuota
        return translated
    except Exception as e:
        return f"[Translation error] {text}"

# Proses translate dan overwrite
for kolom in kolom_terjemahkan:
    if kolom in df.columns:
        df[kolom] = df[kolom].apply(translate_text)
    else:
        print(f"Kolom '{kolom}' tidak ditemukan.")

# Simpan ke file baru (hasil full dalam Bahasa Inggris)
df.to_excel(r"C:\Users\asus\Downloads\wlewle\Data\ITS_IS.xlsx", index=False)
print("✅ File terjemahan berhasil disimpan sebagai ITS_IS.xlsx")


✅ File terjemahan berhasil disimpan sebagai ITS_IS.xlsx
